In [0]:
!pip install pyairbnb

In [0]:
import pyairbnb
from datetime import datetime, timedelta

# 1. Define the search parameters
currency = "INR"            # Indian Rupees
# Note: Using tomorrow's date for discovery (API may not return results for today)
today = datetime.now().date()
tomorrow = today + timedelta(days=1)
check_in = tomorrow.strftime('%Y-%m-%d')
check_out = (tomorrow + timedelta(days=1)).strftime('%Y-%m-%d')

# Bounding box coordinates specifically for T. Nagar, Chennai
ne_lat = 13.0550            # North-East latitude
ne_long = 80.2500           # North-East longitude
sw_lat = 13.0280            # South-West latitude
sw_long = 80.2250           # South-West longitude


zoom_value = 14             # Zoom level for the map (adjust for wider/tighter search)

# Optional Filters (set to 0 or empty to ignore)
price_min = 0
price_max = 0
place_type = ""             # E.g., "Private room" or "Entire home/apt"
amenities = []              # E.g., [4, 7] for WiFi and Pool
free_cancellation = False

# 2. Call the search_all function
try:
    results = pyairbnb.search_all(
        check_in=check_in,
        check_out=check_out,
        ne_lat=ne_lat,
        ne_long=ne_long,
        sw_lat=sw_lat,
        sw_long=sw_long,
        zoom_value=zoom_value,
        price_min=price_min,
        price_max=price_max,
        place_type=place_type,
        amenities=amenities,
        free_cancellation=free_cancellation,
        currency=currency,
        language="en",
        proxy_url=""        # Add a proxy URL here if you are getting blocked by Airbnb
    )
    print(f"Successfully retrieved {len(results)} listings from T. Nagar")

except Exception as e:
    print('errro')
    #print(f"An error occurred: {e}”)

In [0]:
# Convert complex nested data to flat structure for Delta table
import pandas as pd
import json
from datetime import datetime
from pyspark.sql.functions import lit

# Add run_dt to track when data was collected
run_dt = datetime.now().date()

# Flatten the data: keep simple fields as columns, convert complex fields to JSON strings
flattened_data = []
for record in results:
    flat_record = {
        'room_id': record.get('room_id'),
        'name': record.get('name'),
        'title': record.get('title'),
        'type': record.get('type'),
        'category': record.get('category'),
        'price': str(record.get('price')),  # Convert to string to avoid type issues
        'rating': str(record.get('rating')),  # Convert to string to avoid type issues
        'coordinates': json.dumps(record.get('coordinates')),  # Convert to JSON string
        'images': json.dumps(record.get('images')),  # Convert to JSON string
        'badges': json.dumps(record.get('badges')),  # Convert to JSON string
        'full_data': json.dumps(record, default=str),  # Keep full JSON for reference
        'run_dt': run_dt  # Track collection date
    }
    flattened_data.append(flat_record)

# Create pandas DataFrame
pandas_df = pd.DataFrame(flattened_data)

# Convert to Spark DataFrame
df = spark.createDataFrame(pandas_df)

# Use MERGE to prevent duplicates on re-run (upsert based on room_id)
try:
    # If table exists, merge
    df.createOrReplaceTempView("new_listings")
    spark.sql("""
        MERGE INTO workspace.default.tnagar_airbnb_listings AS target
        USING new_listings AS source
        ON target.room_id = source.room_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"Successfully merged {df.count()} Airbnb listings into workspace.default.tnagar_airbnb_listings")
except:
    # If table doesn't exist, create it
    df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.tnagar_airbnb_listings")
    print(f"Successfully created workspace.default.tnagar_airbnb_listings with {df.count()} listings")

display(df.limit(5))

In [0]:
df.display()

In [0]:
# Filter records that have badges (non-empty badges array)
from pyspark.sql.functions import col

df = df.filter(col("badges") != "[]")
print(f"Filtered to {df.count()} listings with badges")
display(df)

In [0]:
# Add Airbnb URL column
from pyspark.sql.functions import concat, lit

df = df.withColumn("airbnb_url", concat(lit("https://www.airbnb.co.in/rooms/"), col("room_id").cast("string")))
print(f"Added airbnb_url column to {df.count()} listings")
display(df.select("room_id", "name", "rating", "badges", "airbnb_url"))

In [0]:
# Extract bedroom information from full_data JSON
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
import json
import re

@udf(returnType=StringType())
def extract_bedroom_info(full_data_str):
    """Extract bedroom and bed information from the full_data JSON"""
    try:
        data = json.loads(full_data_str)
        bed_info = []
        
        # Check structuredContent.mapPrimaryLine for bedroom/bed details
        if 'structuredContent' in data and data['structuredContent']:
            sc = data['structuredContent']
            if 'mapPrimaryLine' in sc and sc['mapPrimaryLine']:
                for item in sc['mapPrimaryLine']:
                    if isinstance(item, dict) and 'body' in item:
                        body = item['body']
                        # Look for bedroom or bed information
                        if 'bed' in body.lower():
                            bed_info.append(body)
        
        # Return combined bedroom info or "Not specified"
        return ', '.join(bed_info) if bed_info else "Not specified"
    except:
        return "Not specified"

# Add bedroom_info column
df = df.withColumn("bedroom_info", extract_bedroom_info(col("full_data")))

print(f"Added bedroom_info column to {df.count()} listings")
display(df.select("room_id", "name", "bedroom_info", "rating", "badges", "airbnb_url"))

In [0]:
# Extract actual bedroom count as a separate numeric column
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType
import re
import json

@udf(returnType=IntegerType())
def extract_bedroom_count(full_data_str):
    """Extract the numeric count of bedrooms directly from JSON mapPrimaryLine"""
    try:
        data = json.loads(full_data_str)
        
        # Check structuredContent.mapPrimaryLine for BEDINFO items
        if 'structuredContent' in data and data['structuredContent']:
            sc = data['structuredContent']
            if 'mapPrimaryLine' in sc and sc['mapPrimaryLine']:
                for item in sc['mapPrimaryLine']:
                    if isinstance(item, dict) and item.get('type') == 'BEDINFO':
                        body = item.get('body', '')
                        # Look for "X bedroom" or "X bedrooms" specifically (not just "X bed")
                        match = re.search(r'(\d+)\s+bedroom', body, re.IGNORECASE)
                        if match:
                            return int(match.group(1))
        
        return None
    except:
        return None

# Add bedroom_count column - extract from full_data JSON
df = df.withColumn("bedroom_count", extract_bedroom_count(col("full_data")))

print(f"Added bedroom_count column to {df.count()} listings")
print("\nBedroom count distribution:")
df.groupBy("bedroom_count").count().orderBy("bedroom_count").show()
print("\nSample data:")
display(df.select("room_id", "name", "bedroom_count", "bedroom_info", "rating", "badges", "airbnb_url").orderBy("bedroom_count"))

In [0]:
# Update the listings table with enriched data (URLs, bedroom info, bedroom count)
print(f"Updating table with enriched data for {df.count()} badged listings...")

# Deduplicate by room_id (keep first occurrence)
df_dedup = df.dropDuplicates(["room_id"])
print(f"Deduplicated to {df_dedup.count()} unique listings")

df_dedup.createOrReplaceTempView("enriched_listings")

spark.sql("""
    MERGE INTO workspace.default.tnagar_airbnb_listings AS target
    USING enriched_listings AS source
    ON target.room_id = source.room_id
    WHEN MATCHED THEN UPDATE SET
        target.airbnb_url = source.airbnb_url,
        target.bedroom_info = source.bedroom_info,
        target.bedroom_count = source.bedroom_count
""")

print("✓ Table updated with enriched data (URLs, bedroom info, bedroom count)")
print("\nVerification - sample enriched records:")
display(spark.table("workspace.default.tnagar_airbnb_listings")
        .select("room_id", "name", "bedroom_count", "bedroom_info", "badges", "airbnb_url")
        .filter(col("badges") != "[]")
        .limit(5))

In [0]:
# Create daily price tracking table schema
from pyspark.sql.types import StructType, StructField, LongType, StringType, DateType, TimestampType, BooleanType, DoubleType
from datetime import datetime, timedelta
import pyairbnb

# Define schema for daily price tracking
daily_price_schema = StructType([
    StructField("room_id", LongType(), False),
    StructField("listing_name", StringType(), True),
    StructField("check_in_date", DateType(), False),
    StructField("check_out_date", DateType(), False),
    StructField("price_per_night", DoubleType(), True),
    StructField("currency", StringType(), True),
    StructField("is_available", BooleanType(), True),
    StructField("scraped_at", TimestampType(), False),
    StructField("year_month", StringType(), False),  # Partition key: YYYY-MM format
    StructField("run_dt", DateType(), False)  # Date when data was collected
])

print("Daily price tracking schema defined")
print("\nTable will track:")
print("  - room_id: Listing identifier")
print("  - listing_name: Property name")
print("  - check_in_date: Date to check availability")
print("  - check_out_date: Following day")
print("  - price_per_night: Price for that night")
print("  - currency: INR")
print("  - is_available: Whether the listing is available or booked")
print("  - scraped_at: Timestamp when data was collected")
print("  - year_month: Partition key for efficient queries")
print("  - run_dt: Date when scrape was executed (for idempotency)")
print("\nTarget table: workspace.default.tnagar_airbnb_daily_prices")
print("\nScraping logic: Starts from TODAY and goes to end of month")
print("MERGE logic ensures no duplicates on re-run (key: room_id + check_in_date)")

In [0]:
# Function to scrape daily prices for a specific date range
import pandas as pd
from datetime import datetime, timedelta
import pyairbnb

def scrape_daily_prices_for_month(room_ids, year, month):
    """
    Scrape price and availability data from TODAY to end of the specified month
    for all given room IDs.
    
    Args:
        room_ids: List of Airbnb room IDs to check
        year: Year (e.g., 2026)
        month: Month (1-12)
    
    Returns:
        List of dictionaries with daily price data
    """
    # Start from today (or month start if today is before the target month)
    today = datetime.now().date()
    month_start = datetime(year, month, 1).date()
    start_date = datetime.combine(max(today, month_start), datetime.min.time())
    if month == 12:
        end_date = datetime(year + 1, 1, 1) - timedelta(days=1)
    else:
        end_date = datetime(year, month + 1, 1) - timedelta(days=1)
    
    daily_records = []
    scraped_at = datetime.now()
    year_month = f"{year}-{month:02d}"
    
    # T. Nagar bounding box
    ne_lat = 13.0550
    ne_long = 80.2500
    sw_lat = 13.0280
    sw_long = 80.2250
    
    print(f"Scraping prices for {len(room_ids)} listings from {start_date.date()} to {end_date.date()} (today onwards)...")
    
    # Iterate through each day of the month
    current_date = start_date
    day_count = 0
    
    while current_date <= end_date:
        check_in = current_date.strftime('%Y-%m-%d')
        check_out = (current_date + timedelta(days=1)).strftime('%Y-%m-%d')
        
        day_count += 1
        print(f"\n  Day {day_count}: {check_in}...")
        
        try:
            # Search Airbnb for this specific date
            results = pyairbnb.search_all(
                check_in=check_in,
                check_out=check_out,
                ne_lat=ne_lat,
                ne_long=ne_long,
                sw_lat=sw_lat,
                sw_long=sw_long,
                zoom_value=14,
                price_min=0,
                price_max=0,
                currency="INR",
                language="en"
            )
            
            # Create a lookup dict for faster access
            results_dict = {r.get('room_id'): r for r in results}
            
            # Check each room_id we're tracking
            for room_id in room_ids:
                if room_id in results_dict:
                    listing = results_dict[room_id]
                    price_info = listing.get('price', {})
                    
                    # Extract price value
                    price_value = None
                    if isinstance(price_info, dict):
                        if 'unit' in price_info and 'amount' in price_info['unit']:
                            price_value = float(price_info['unit']['amount'])
                    
                    daily_records.append({
                        'room_id': room_id,
                        'listing_name': listing.get('name', ''),
                        'check_in_date': current_date.date(),
                        'check_out_date': (current_date + timedelta(days=1)).date(),
                        'price_per_night': price_value,
                        'currency': 'INR',
                        'is_available': True,  # If it appears in results, it's available
                        'scraped_at': scraped_at,
                        'year_month': year_month
                    })
                else:
                    # Room not in results = likely booked/unavailable
                    daily_records.append({
                        'room_id': room_id,
                        'listing_name': None,  # We don't have the name if it's not in results
                        'check_in_date': current_date.date(),
                        'check_out_date': (current_date + timedelta(days=1)).date(),
                        'price_per_night': None,
                        'currency': 'INR',
                        'is_available': False,  # Not available = booked
                        'scraped_at': scraped_at,
                        'year_month': year_month
                    })
            
            print(f"    Processed {len(room_ids)} listings")
            
        except Exception as e:
            print(f"    Error scraping {check_in}: {e}")
        
        current_date += timedelta(days=1)
    
    print(f"\nTotal records created: {len(daily_records)}")
    return daily_records

print("Daily price scraping function defined")

In [0]:
# Run batch process to scrape August 2026 prices (from today onwards) for all badged listings
from datetime import datetime

# Get the room IDs from our badged listings table
room_ids_to_track = [row.room_id for row in spark.table("workspace.default.tnagar_airbnb_listings").select("room_id").distinct().collect()]

print(f"Tracking {len(room_ids_to_track)} listings with badges")
print(f"Room IDs: {room_ids_to_track[:5]}... (showing first 5)")

# Scrape for August 2026 (starting from today's date to end of month)
print("\n" + "="*60)
print(f"Starting batch scrape for August 2026 (from today: {datetime.now().date()} onwards)...")
print("="*60)

daily_data = scrape_daily_prices_for_month(
    room_ids=room_ids_to_track,
    year=2026,
    month=8
)

print("\n" + "="*60)
print("Batch scrape completed!")
print(f"Total records collected: {len(daily_data)}")
print("="*60)

In [0]:
# Save daily price data to Delta table
import pandas as pd
from datetime import datetime
from pyspark.sql.types import StructType, StructField, LongType, StringType, DateType, TimestampType, BooleanType, DoubleType
from pyspark.sql.functions import lit

# Add run_dt to track when data was collected
run_dt = datetime.now().date()

# Add run_dt to each record
for record in daily_data:
    record['run_dt'] = run_dt

# Convert to pandas DataFrame
daily_df_pandas = pd.DataFrame(daily_data)

# Convert to Spark DataFrame with explicit schema
daily_price_schema = StructType([
    StructField("room_id", LongType(), False),
    StructField("listing_name", StringType(), True),
    StructField("check_in_date", DateType(), False),
    StructField("check_out_date", DateType(), False),
    StructField("price_per_night", DoubleType(), True),
    StructField("currency", StringType(), True),
    StructField("is_available", BooleanType(), True),
    StructField("scraped_at", TimestampType(), False),
    StructField("year_month", StringType(), False),
    StructField("run_dt", DateType(), False)
])

daily_df_spark = spark.createDataFrame(daily_df_pandas, schema=daily_price_schema)

# Use MERGE to prevent duplicates on re-run (upsert based on room_id + check_in_date)
try:
    # If table exists, merge
    daily_df_spark.createOrReplaceTempView("new_daily_prices")
    spark.sql("""
        MERGE INTO workspace.default.tnagar_airbnb_daily_prices AS target
        USING new_daily_prices AS source
        ON target.room_id = source.room_id 
           AND target.check_in_date = source.check_in_date
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"✓ Merged {daily_df_spark.count()} daily price records into workspace.default.tnagar_airbnb_daily_prices")
except:
    # If table doesn't exist, create it
    daily_df_spark.write.format("delta") \
        .mode("overwrite") \
        .partitionBy("year_month") \
        .saveAsTable("workspace.default.tnagar_airbnb_daily_prices")
    print(f"✓ Created workspace.default.tnagar_airbnb_daily_prices with {daily_df_spark.count()} records")

# Show sample data
print("\nSample of daily price data:")
display(spark.table("workspace.default.tnagar_airbnb_daily_prices").orderBy("room_id", "check_in_date").limit(10))

In [0]:
%sql
-- COMPLETE MICRO-LEVEL DATA: Daily prices joined with enriched listing details
SELECT 
  -- Listing identification
  dp.room_id,
  l.name as listing_name,
  l.title as listing_type,
  
  -- Listing features (from enriched listings table)
  l.bedroom_count,
  l.bedroom_info,
  l.badges,
  l.rating,
  l.airbnb_url,
  
  -- Daily availability & pricing
  dp.check_in_date,
  DAYOFWEEK(dp.check_in_date) as day_of_week,
  CASE DAYOFWEEK(dp.check_in_date)
    WHEN 1 THEN 'Sunday'
    WHEN 2 THEN 'Monday'
    WHEN 3 THEN 'Tuesday'
    WHEN 4 THEN 'Wednesday'
    WHEN 5 THEN 'Thursday'
    WHEN 6 THEN 'Friday'
    WHEN 7 THEN 'Saturday'
  END as day_name,
  dp.is_available,
  CASE 
    WHEN dp.is_available = TRUE THEN 'Available'
    ELSE 'Booked'
  END as status,
  dp.price_per_night,
  dp.currency,
  
  -- Metadata
  dp.scraped_at,
  dp.run_dt as data_collection_date
  
FROM workspace.default.tnagar_airbnb_daily_prices dp
LEFT JOIN workspace.default.tnagar_airbnb_listings l
  ON dp.room_id = l.room_id
ORDER BY dp.room_id, dp.check_in_date
LIMIT 100;

In [0]:
%sql
-- ANALYSIS 1: Booking patterns by bedroom count and badges
SELECT 
  l.bedroom_count,
  l.badges,
  l.rating,
  COUNT(DISTINCT dp.room_id) as listing_count,
  COUNT(*) as total_days_tracked,
  SUM(CASE WHEN dp.is_available = FALSE THEN 1 ELSE 0 END) as booked_days,
  SUM(CASE WHEN dp.is_available = TRUE THEN 1 ELSE 0 END) as available_days,
  ROUND(SUM(CASE WHEN dp.is_available = FALSE THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) as booking_rate_pct,
  ROUND(AVG(dp.price_per_night), 2) as avg_price_when_available
FROM workspace.default.tnagar_airbnb_daily_prices dp
LEFT JOIN workspace.default.tnagar_airbnb_listings l
  ON dp.room_id = l.room_id
GROUP BY l.bedroom_count, l.badges, l.rating
ORDER BY booking_rate_pct DESC;

In [0]:
%sql
-- ANALYSIS 2: Booking patterns by property type and day of week
SELECT 
  CASE 
    WHEN l.bedroom_count IS NULL THEN 'Studio/Room'
    WHEN l.bedroom_count = 1 THEN '1 Bedroom'
    WHEN l.bedroom_count = 2 THEN '2 Bedrooms'
    WHEN l.bedroom_count = 3 THEN '3 Bedrooms'
    ELSE '4+ Bedrooms'
  END as property_type,
  CASE DAYOFWEEK(dp.check_in_date)
    WHEN 1 THEN 'Sunday'
    WHEN 2 THEN 'Monday'
    WHEN 3 THEN 'Tuesday'
    WHEN 4 THEN 'Wednesday'
    WHEN 5 THEN 'Thursday'
    WHEN 6 THEN 'Friday'
    WHEN 7 THEN 'Saturday'
  END as day_name,
  COUNT(*) as total_listings_checked,
  SUM(CASE WHEN dp.is_available = FALSE THEN 1 ELSE 0 END) as booked,
  SUM(CASE WHEN dp.is_available = TRUE THEN 1 ELSE 0 END) as available,
  ROUND(SUM(CASE WHEN dp.is_available = FALSE THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) as booking_rate_pct
FROM workspace.default.tnagar_airbnb_daily_prices dp
LEFT JOIN workspace.default.tnagar_airbnb_listings l
  ON dp.room_id = l.room_id
GROUP BY 
  CASE 
    WHEN l.bedroom_count IS NULL THEN 'Studio/Room'
    WHEN l.bedroom_count = 1 THEN '1 Bedroom'
    WHEN l.bedroom_count = 2 THEN '2 Bedrooms'
    WHEN l.bedroom_count = 3 THEN '3 Bedrooms'
    ELSE '4+ Bedrooms'
  END,
  CASE DAYOFWEEK(dp.check_in_date)
    WHEN 1 THEN 'Sunday'
    WHEN 2 THEN 'Monday'
    WHEN 3 THEN 'Tuesday'
    WHEN 4 THEN 'Wednesday'
    WHEN 5 THEN 'Thursday'
    WHEN 6 THEN 'Friday'
    WHEN 7 THEN 'Saturday'
  END
ORDER BY property_type, 
  CASE day_name
    WHEN 'Monday' THEN 1
    WHEN 'Tuesday' THEN 2
    WHEN 'Wednesday' THEN 3
    WHEN 'Thursday' THEN 4
    WHEN 'Friday' THEN 5
    WHEN 'Saturday' THEN 6
    WHEN 'Sunday' THEN 7
  END;

In [0]:
%sql
-- ANALYSIS 3: Sample detailed listing profiles with all daily data
-- (Showing only badged listings with complete data including URLs)
WITH sample_listings AS (
  SELECT DISTINCT room_id
  FROM workspace.default.tnagar_airbnb_listings
  WHERE badges != '[]' AND airbnb_url IS NOT NULL
  ORDER BY room_id
  LIMIT 3
)
SELECT 
  l.room_id,
  l.name as listing_name,
  l.bedroom_count,
  l.bedroom_info,
  l.badges,
  l.rating,
  l.airbnb_url,
  dp.check_in_date,
  CASE DAYOFWEEK(dp.check_in_date)
    WHEN 1 THEN 'Sun'
    WHEN 2 THEN 'Mon'
    WHEN 3 THEN 'Tue'
    WHEN 4 THEN 'Wed'
    WHEN 5 THEN 'Thu'
    WHEN 6 THEN 'Fri'
    WHEN 7 THEN 'Sat'
  END as day,
  CASE 
    WHEN dp.is_available = TRUE THEN '✓ Available'
    ELSE '✗ Booked'
  END as status,
  dp.price_per_night
FROM workspace.default.tnagar_airbnb_daily_prices dp
INNER JOIN workspace.default.tnagar_airbnb_listings l
  ON dp.room_id = l.room_id
WHERE dp.room_id IN (SELECT room_id FROM sample_listings)
ORDER BY l.room_id, dp.check_in_date;

# 🎯 Complete Micro-Level Airbnb Analytics System

## 📊 Data Architecture

### Two Core Tables:

**1. Listings Table:** `workspace.default.tnagar_airbnb_listings` (15 badged listings)
- `room_id`, `name`, `title`, `type`, `category`
- `bedroom_count` (1, 2, 3 bedrooms or null for Studio/Room)
- `bedroom_info` (detailed bed configuration)
- `badges` (SUPERHOST, GUEST_FAVORITE, TOP_X_GUEST_FAVORITE)
- `rating` (score + review count)
- `airbnb_url` (direct link to listing)
- `price`, `coordinates`, `images`, `full_data`
- `run_dt` (data collection date)

**2. Daily Prices Table:** `workspace.default.tnagar_airbnb_daily_prices` (465 daily records)
- `room_id`, `listing_name`
- `check_in_date`, `check_out_date`
- `price_per_night`, `currency`
- `is_available` (TRUE = available, FALSE = booked)
- `scraped_at`, `year_month`, `run_dt`

### 🔗 Joined View: Complete Micro-Level Data

**Cell 13-16** demonstrate the full JOIN showing:
- Every listing's daily status (465 rows = 15 listings × 31 days)
- Each row = one property + one specific date
- All listing features (bedrooms, badges, ratings, URLs)
- Daily availability and pricing
- Day-of-week analysis

## 💡 Key Insights Available

### From Cell 14: Booking Rates by Property Features
- **TOP_X_GUEST_FAVORITE studio:** 90.3% booking rate (highest)
- **GUEST_FAVORITE 1-bedroom:** 80.6% booking rate
- **SUPERHOST room:** 71% booking rate

### From Cell 15: Weekend vs Weekday Patterns
- **3-bedroom properties:** Peak on weekends (60% Sun, 46.7% Sat)
- **Studios/Rooms:** Busiest on Sundays (53.3%)
- **1-bedroom:** Consistent 35-45% across all days

### From Cell 16: Daily Tracking Examples
- **TNagar Room 19755457:** Booked solid Aug 2-23 (22 nights), available Aug 24-31
- **Ravenala Flat:** 80.6% booking rate through August
- **Basera56 Room 4:** 90.3% booking rate - almost fully booked

## 🔄 How the System Works

1. **Cells 1-2:** Install pyairbnb + scrape T. Nagar listings
2. **Cell 3:** Store listings in Delta table with MERGE logic
3. **Cells 5-8:** Filter for badges + enrich with bedroom/URL data
4. **Cells 9-12:** Scrape daily prices for entire month + MERGE to table
5. **Cells 13-16:** Comprehensive joined analysis with all attributes
6. **MERGE logic:** Ensures no duplicates on re-run (idempotent)

## 📅 Running as Scheduled Job

**To automate daily tracking:**

1. **Schedule this notebook** to run daily via Databricks Jobs
2. **Modify Cell 11:** Change to scrape only tomorrow's date:
   ```python
   # Instead of scraping whole month, scrape just tomorrow
   tomorrow = datetime.now() + timedelta(days=1)
   daily_data = scrape_daily_prices_for_month(
       room_ids=room_ids_to_track,
       year=tomorrow.year,
       month=tomorrow.month
   )
   ```
3. **MERGE logic in Cell 12** will append new data without duplicates
4. Historical data accumulates over time

## 🎯 Use Cases

✅ **Demand Forecasting:** Predict booking rates by property type and season
✅ **Price Optimization:** Track price fluctuations for optimal pricing strategy
✅ **Competitive Analysis:** Compare badges, bedrooms, ratings vs booking rates
✅ **Revenue Planning:** Identify high-demand periods and property types
✅ **Booking Patterns:** Weekend vs weekday trends by property size
✅ **Investment Decisions:** Which property types/badges drive highest occupancy

## 🔍 Query Examples

**Find all available 2-bedroom properties for a specific date:**
```sql
SELECT * FROM workspace.default.tnagar_airbnb_daily_prices dp
JOIN workspace.default.tnagar_airbnb_listings l ON dp.room_id = l.room_id
WHERE l.bedroom_count = 2 AND dp.check_in_date = '2026-08-15' AND dp.is_available = TRUE;
```

**Track booking streak for a specific listing:**
```sql
SELECT check_in_date, is_available, status
FROM workspace.default.tnagar_airbnb_daily_prices
WHERE room_id = 19755457
ORDER BY check_in_date;
```

In [0]:
%sql
-- August 2026 Booking Summary: Days Booked, Booking %, Avg Price by Listing
SELECT 
  l.room_id,
  l.name as listing_name,
  l.airbnb_url,
  
  -- Property details
  COALESCE(l.bedroom_count, 0) as bedroom_count,
  l.bedroom_info,
  CASE 
    WHEN l.bedroom_count IS NULL THEN 'Studio/Room'
    WHEN l.bedroom_count = 1 THEN '1 Bedroom'
    WHEN l.bedroom_count = 2 THEN '2 Bedrooms'
    WHEN l.bedroom_count = 3 THEN '3 Bedrooms'
    ELSE '4+ Bedrooms'
  END as property_type,
  l.badges,
  l.rating,
  
  -- Booking metrics for August 2026
  COUNT(*) as total_days_in_month,
  SUM(CASE WHEN dp.is_available = FALSE THEN 1 ELSE 0 END) as days_booked,
  SUM(CASE WHEN dp.is_available = TRUE THEN 1 ELSE 0 END) as days_available,
  ROUND(SUM(CASE WHEN dp.is_available = FALSE THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) as pct_days_booked,
  
  -- Price metrics (only for available days)
  ROUND(AVG(CASE WHEN dp.price_per_night IS NOT NULL THEN dp.price_per_night END), 0) as avg_price_per_night,
  CONCAT('₹', FORMAT_NUMBER(AVG(CASE WHEN dp.price_per_night IS NOT NULL THEN dp.price_per_night END), 0)) as formatted_avg_price,
  ROUND(MIN(CASE WHEN dp.price_per_night IS NOT NULL THEN dp.price_per_night END), 0) as min_price,
  ROUND(MAX(CASE WHEN dp.price_per_night IS NOT NULL THEN dp.price_per_night END), 0) as max_price,
  
  -- Revenue potential (days available × avg price)
  ROUND(SUM(CASE WHEN dp.is_available = TRUE THEN 1 ELSE 0 END) * 
        AVG(CASE WHEN dp.price_per_night IS NOT NULL THEN dp.price_per_night END), 0) as potential_revenue
  
FROM workspace.default.tnagar_airbnb_daily_prices dp
LEFT JOIN workspace.default.tnagar_airbnb_listings l
  ON dp.room_id = l.room_id
WHERE dp.year_month = '2026-08'
  AND l.badges != '[]'  -- Filter for badged listings only (those with URLs)
GROUP BY 
  l.room_id, 
  l.name, 
  l.airbnb_url,
  l.bedroom_count,
  l.bedroom_info,
  l.badges,
  l.rating
ORDER BY pct_days_booked DESC;